# Introduction
The prediction of stock market movements has been a challenging task in financial research.
Traditional models rely on historical price data and fundamental analysis, but recent studies
suggest that social media, particularly Twitter, significantly influences market trends. However,
most sentiment-based approaches treat all tweets equally, overlooking the fact that not all tweets
have the same impact. This paper proposes integrating social network analysis (SNA) into stock
price prediction by considering network-related metrics such as degree centrality, PageRank,
and betweenness centrality.

# Research Idea
The research aims to answer the following question: How can social network analysis improve
the predictive power of stock market models using Twitter data? Previous studies have used
sentiment analysis to extract public opinion, but this approach does not account for the varying
influence of different users. This study introduces network-based features, leveraging retweets,
mentions, and likes to determine the influence of a tweet.

# Network Analysis in Twitter Data
Social network analysis allows us to quantify the influence of individual tweets using:

**• Degree Centrality:**
 Measures how many connections (mentions or retweets) a user has,
indicating their influence.

**• PageRank:** 
Identifies authoritative users whose tweets may have a greater impact on
stock movements.

**• Betweenness Centrality:**
 Highlights users who bridge different communities, potentially
affecting information flow.
These metrics provide deeper insights into the spread and impact of financial opinions on Twit-
ter.

In [44]:
import pandas as pd
import networkx as nx

# Load your tweet dataset
df_full = pd.read_csv('/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/src/models/merged_sentiment_output.csv')  # Contains tweet_id, writer, comment_num, retweet_num, like_num

# Create directed graph
G = nx.DiGraph()

In [45]:
df_net = df_full[df_full['writer'].notnull()]

In [46]:
# Iterate over tweets and add edges from writer to "tweet engagement"
for idx, row in df_net.iterrows():
    user = row['writer']
    tweet_id = row['tweet_id']
    
    # Use engagement as weight (tune as needed)
    weight = row.get('retweet_num', 0) + row.get('like_num', 0) + row.get('comment_num', 0)
    
    # If you want to connect users, simulate self-loop or user-tweet interaction
    if weight > 0:
        G.add_edge(user, f"tweet_{tweet_id}", weight=weight)


In [47]:
import re

for idx, row in df_net.iterrows():
    writer = row['writer']
    text = row['text']
    mentions = re.findall(r'@(\w+)', text) if pd.notna(text) else []

    for mentioned_user in mentions:
        weight = row.get('retweet_num', 0) + row.get('like_num', 0) + row.get('comment_num', 0)
        G.add_edge(writer, mentioned_user, weight=weight)


In [48]:
# Degree Centrality (weighted)
degree_centrality = nx.degree_centrality(G)

# PageRank (weighted by engagement)
pagerank = nx.pagerank(G, weight='weight')

# Betweenness Centrality (can be expensive; use approx for large graphs)
betweenness = nx.betweenness_centrality(G, weight='weight', k=500)  # 'k' = number of node samples


In [62]:
df_net.isnull().sum()

tweet_id                   0
writer                     0
created_at                 0
text                       0
comment_num                0
retweet_num                0
like_num                   0
Text_Cleaned               0
sentiment                  0
degree_centrality          0
pagerank                   0
betweenness                0
sentiment_mapped      363637
weighted_sentiment    363637
pagerank_bin               0
pagerank_weight            0
dtype: int64

In [63]:
df_net['degree_centrality'] = df_net['writer'].map(degree_centrality).fillna(0)
df_net['pagerank'] = df_net['writer'].map(pagerank).fillna(0)
df_net['betweenness'] = df_net['writer'].map(betweenness).fillna(0)


/var/folders/qz/ctfh_y5s4f5g3k2h_nzprjm00000gn/T/ipykernel_35397/950737450.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_net['degree_centrality'] = df_net['writer'].map(degree_centrality).fillna(0)
/var/folders/qz/ctfh_y5s4f5g3k2h_nzprjm00000gn/T/ipykernel_35397/950737450.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_net['pagerank'] = df_net['writer'].map(pagerank).fillna(0)
/var/folders/qz/ctfh_y5s4f5g3k2h_nzprjm00000gn/T/ipykernel_35397/950737450.py:3: SettingWithCopyWarning: 
A value is

In [72]:
df_net.head()

,tweet_id,writer,created_at,text,comment_num,retweet_num,like_num,Text_Cleaned,sentiment,degree_centrality,pagerank,betweenness,sentiment_mapped,weighted_sentiment,pagerank_bin,pagerank_weight,pagerank_scaled,weighted_sentiment_scaled
0,797587806331830273,NawazGafar,2016-11-12 23:52:06,So there might be a correlation between Trump ...,0,0,0,correlation trump black deals,negative,0.000007,5.866570e-07,0.0,-1,-0.000163,0,0.000000,0.000163,-0.000163
1,797589836278464512,iHotStockPicks,2016-11-13 00:00:10,$ROKA 1m float and low rsi going to supernova ...,0,0,1,roka m float low rsi supernova monday hmny mfs...,negative,0.000174,5.866570e-07,0.0,-1,-0.000163,0,0.000000,0.000163,-0.000163
2,797590101140340736,Karla_Tango,2016-11-13 00:01:13,$AAPL stock should take a hitTrump said he mak...,2,1,3,hittrump bring labor higher coststocks ridiculous,positive,0.000012,5.988003e-07,0.0,1,0.000167,2,0.333333,0.000167,0.000167
4,797591064081301536,StocksBomb,2016-11-13 00:05:02,"$ROKA is Monday's SUPERNOVA. GET IN, only 1M F...",0,0,0,roka mondays supernova m float run volume nvda...,positive,0.000141,5.866570e-07,0.0,1,0.000163,0,0.000000,0.000163,0.000163
5,797591529908883456,IndigoBlueUSA,2016-11-13 00:06:53,@Tim_Cook Can you explain why $AAPL @Apple sti...,0,0,0,timcook explain moved csuite diversity forward...,positive,0.000100,5.899410e-07,0.0,1,0.000164,1,0.166667,0.000164,0.000164


In [65]:
print(df_net.columns)


Index(['tweet_id', 'writer', 'created_at', 'text', 'comment_num',
       'retweet_num', 'like_num', 'Text_Cleaned', 'sentiment',
       'degree_centrality', 'pagerank', 'betweenness', 'sentiment_mapped',
       'weighted_sentiment', 'pagerank_bin', 'pagerank_weight'],
      dtype='object')


In [66]:
print("Min:", df_net['pagerank'].min())
print("Max:", df_net['pagerank'].max())


Min: 0.0
Max: 0.0035897231188468885


In [67]:
df_net['pagerank'].value_counts()

pagerank
5.866570e-07    1447570
5.910083e-07     182392
8.477390e-07     111954
1.374166e-06      91741
9.130095e-07      59838
                 ...   
1.133645e-06          1
5.873684e-07          1
6.155302e-07          1
6.223514e-07          1
9.099361e-07          1
Name: count, Length: 6423, dtype: int64

In [68]:
df_net['sentiment'] = df_net['sentiment'].str.lower()


/var/folders/qz/ctfh_y5s4f5g3k2h_nzprjm00000gn/T/ipykernel_35397/3812448157.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_net['sentiment'] = df_net['sentiment'].str.lower()


In [70]:
from sklearn.preprocessing import MinMaxScaler
sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
df_net['sentiment_mapped'] = df_net['sentiment'].map(sentiment_map)

df_net['pagerank_scaled'] = MinMaxScaler().fit_transform(df_net[['pagerank']])
df_net['weighted_sentiment'] = df_net['pagerank_scaled'] * df_net['sentiment_mapped']


/var/folders/qz/ctfh_y5s4f5g3k2h_nzprjm00000gn/T/ipykernel_35397/2010368435.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_net['sentiment_mapped'] = df_net['sentiment'].map(sentiment_map)
/var/folders/qz/ctfh_y5s4f5g3k2h_nzprjm00000gn/T/ipykernel_35397/2010368435.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_net['pagerank_scaled'] = MinMaxScaler().fit_transform(df_net[['pagerank']])
/var/folders/qz/ctfh_y5s4f5g3k2h_nzprjm00000gn/T/ipykernel_35397/2010368435.py:6: SettingWithCopyWarning: 
A 

In [73]:
df_net.head()

,tweet_id,writer,created_at,text,comment_num,retweet_num,like_num,Text_Cleaned,sentiment,degree_centrality,pagerank,betweenness,sentiment_mapped,weighted_sentiment,pagerank_bin,pagerank_weight,pagerank_scaled,weighted_sentiment_scaled
0,797587806331830273,NawazGafar,2016-11-12 23:52:06,So there might be a correlation between Trump ...,0,0,0,correlation trump black deals,negative,0.000007,5.866570e-07,0.0,-1,-0.000163,0,0.000000,0.000163,-0.000163
1,797589836278464512,iHotStockPicks,2016-11-13 00:00:10,$ROKA 1m float and low rsi going to supernova ...,0,0,1,roka m float low rsi supernova monday hmny mfs...,negative,0.000174,5.866570e-07,0.0,-1,-0.000163,0,0.000000,0.000163,-0.000163
2,797590101140340736,Karla_Tango,2016-11-13 00:01:13,$AAPL stock should take a hitTrump said he mak...,2,1,3,hittrump bring labor higher coststocks ridiculous,positive,0.000012,5.988003e-07,0.0,1,0.000167,2,0.333333,0.000167,0.000167
4,797591064081301536,StocksBomb,2016-11-13 00:05:02,"$ROKA is Monday's SUPERNOVA. GET IN, only 1M F...",0,0,0,roka mondays supernova m float run volume nvda...,positive,0.000141,5.866570e-07,0.0,1,0.000163,0,0.000000,0.000163,0.000163
5,797591529908883456,IndigoBlueUSA,2016-11-13 00:06:53,@Tim_Cook Can you explain why $AAPL @Apple sti...,0,0,0,timcook explain moved csuite diversity forward...,positive,0.000100,5.899410e-07,0.0,1,0.000164,1,0.166667,0.000164,0.000164


In [71]:
# Keep only the desired columns
columns_to_keep = ['tweet_id', 'created_at', 'sentiment','weighted_sentiment']
df_selected = df_net[columns_to_keep].copy()

# Save the selected data to a CSV file
df_selected.to_csv("pagerank_weighted_sentiment.csv", index=False)

print("File saved as pagerank_weighted_sentiment.csv")

File saved as pagerank_weighted_sentiment.csv


**1. Katz Centrality**

Category: Node Influence & Centrality
Summary:
Katz centrality measures a node's influence by considering both direct connections and indirect paths through the network, assigning more weight to closer nodes. It's useful in identifying key influencers in a network (e.g., top users, critical assets).

**2. Clustering on Network Data**

Category: Community Detection & Group Analysis
Summary:
Clustering in networks groups nodes into communities where internal connections are denser than external ones. Methods include Louvain, spectral clustering, and label propagation. These are used to detect communities, segments, or behavioral clusters.

**3. Connectivity**

Category: Network Structure & Robustness
Summary:
Connectivity evaluates how well nodes are linked in a network. This includes node/edge connectivity, connected components, and path-based measures. It's crucial for understanding the resilience of networks and flow of information or influence.

**4. Thesis Integration**

*Category:* Application in Research
*Summary:*
Explained how to apply Katz centrality, clustering, and connectivity in your thesis:

Katz: for influence scoring

Clustering: for community analysis

Connectivity: for robustness and structure evaluation
Each can form part of the methodology, analysis, and results discussion in your research.